In [1]:
from hashlib import sha256
import json
from pathlib import Path
import sys
from tempfile import TemporaryDirectory, mkdtemp

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if not (ROOT / 'streamer_drift_diffusion').is_dir():
    ROOT = ROOT / 'pyaxistreamer-master'
if not (ROOT / 'streamer_drift_diffusion').is_dir():
    raise RuntimeError('pyaxistreamer-master またはその親を作業フォルダにしてください')
sys.path.insert(0, str(ROOT))
from streamer_drift_diffusion import SimulationConfig, run_simulation
from streamer_drift_diffusion.kinetics import MECHANISM_PATH
from streamer_drift_diffusion.plasma_chemistry import generate_reaction_table, load_reaction_table
import fipy
print('Python:', sys.executable)
print('FiPy:', fipy.__version__)

Python: c:\Users\daiko\Documents\Python_venvSystem\3.14system\PyAxiStreamer\env_pyaxistreamer\Scripts\python.exe
FiPy: 4.0.3


In [2]:
# 使用するBOLSIG+形式の断面積ファイル。ここではPy3BOLOS付属のPhelpsデータを指定。
CROSS_SECTIONS = ROOT.parent / 'Py3bolos' / 'Phelps' / 'Phelps_N2-O2_20261026.txt'
TABLE_PATH = ROOT / 'outputdata_coefficients' / 'air_mechanism_rates.json'
FIELD_TD = np.geomspace(0.01, 2000., 257)  # 0 Tdは生成時に追加される。
REBUILD_TABLE = False  # 断面積・反応機構・電場範囲・温度・組成を変更したときはTrue。

# 形式2の係数表からBOLOSの移動度・拡散とCanteraの二体反応係数を使う。
config = SimulationConfig(
    nr=400, nz=300, delta=5e-6, electrode_gap=1e-3,
    chemistry_table=TABLE_PATH,
    voltage_mode='impulse', voltage=4000.0,
    impulse_peak_voltage=6000.0, impulse_time_scale=0.25,
    adaptive_timestep=True, t_end=5.001e-9,
    output_interval=1e-10, dt_max=1e-12,
    max_steps=20_000,
)
OUTPUT_BASE = ROOT / ('outputdata_chemistry2' if config.adaptive_timestep else 'outputdata_staggered')
print(config)

SimulationConfig(nr=400, nz=300, delta=5e-06, temperature=300.0, avogadro=6.022141e+23, gas_constant=0.08205736608096, elementary_charge=1.602176462e-19, vacuum_permittivity=8.85e-12, oxygen_fraction=0.2, nitrogen_fraction=0.8, cathode_radius=0.0003, voltage=4000.0, voltage_mode='impulse', impulse_peak_voltage=6000.0, impulse_time_scale=0.25, voltage_waveform_file='C:\\Users\\daiko\\Documents\\Python_venvSystem\\3.14system\\PyAxiStreamer\\pyaxistreamer-master\\streamer_drift_diffusion\\data\\impulse_waveform_V50_tr5_tw6.txt', dt=1e-12, nmax=1e+19, z0=0.0009, sr0=0.0001, sz0=0.0001, background_ionization=10000000000000.0, poisson_tolerance=1e-10, adaptive_timestep=True, t_end=5.001e-09, dt_max=1e-12, dt_min=1e-18, dt_safety_factor=0.5, dt_cfl_number=0.8, dt_chemistry_fraction=0.1, dt_growth_factor=1.2, max_step_retries=12, max_steps=20000, output_interval=1e-10, electrode_gap=0.001, electron_diffusion=0.0, chemistry_table=WindowsPath('c:/Users/daiko/Documents/Python_venvSystem/3.14syste

In [3]:
# 初回は係数表を生成し、次回は同じ入力で作成済みの表を再利用する。
if not CROSS_SECTIONS.is_file():
    raise FileNotFoundError(f'CROSS_SECTIONS に断面積ファイルを指定してください: {CROSS_SECTIONS}')
source_hash = sha256(CROSS_SECTIONS.read_bytes()).hexdigest()
if REBUILD_TABLE or not TABLE_PATH.is_file():
    print('Py3BOLOSとCanteraで係数表を生成しています:', CROSS_SECTIONS, flush=True)
    data = generate_reaction_table(
        CROSS_SECTIONS, FIELD_TD, temperature=config.temperature,
        nitrogen_fraction=config.nitrogen_fraction,
    )
    TABLE_PATH.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=TABLE_PATH.parent) as directory:
        pending = Path(directory) / TABLE_PATH.name
        pending.write_text(json.dumps(data, indent=2, allow_nan=False) + '\n', encoding='utf-8')
        pending.replace(TABLE_PATH)

table = load_reaction_table(TABLE_PATH, config)
if (table.mechanism is None or table.metadata.get('source_sha256') != source_hash
        or table.metadata.get('mechanism_sha256') != sha256(MECHANISM_PATH.read_bytes()).hexdigest()
        or not np.array_equal(table.field_td[1:], FIELD_TD)):
    raise ValueError('係数表と入力条件が一致しません。REBUILD_TABLE=True にして設定セルから実行してください。')
print('係数表:', TABLE_PATH)
print(f'E/N: {table.field_td[0]:g} ～ {table.field_td[-1]:g} Td ({table.field_td.size}点)')

Py3BOLOSとCanteraで係数表を生成しています: c:\Users\daiko\Documents\Python_venvSystem\3.14system\PyAxiStreamer\Py3bolos\Phelps\Phelps_N2-O2_20261026.txt


Target N2: EFFECTIVE minus INELASTIC is negative at 6 table nodes (1500..10000 eV); clipped to zero. Interpolated ELASTIC may be affected within 1000..10000 eV. This can reflect constant extrapolation beyond individual tables, not necessarily invalid source data.
EEDF tail slopes undefined in 1 occupied cells (energy 19.1427..19.1427 eV; discrete population fraction 4.82e-320). Legacy zero-integral fallback retained; check grid/cutoff convergence before ignoring.
EEDF tail slopes undefined in 4 occupied cells (energy 17.627..18.6814 eV; discrete population fraction 3.77e-18). Legacy zero-integral fallback retained; check grid/cutoff convergence before ignoring.
Negative EEDF values during iteration: relative negative population 1.2e-11; verify the final EEDF and rates.
EEDF tail slopes undefined in 39 occupied cells (energy 0.144531..444.393 eV; discrete population fraction 4.08e-12). Legacy zero-integral fallback retained; check grid/cutoff convergence before ignoring.
Negative EEDF v

係数表: c:\Users\daiko\Documents\Python_venvSystem\3.14system\PyAxiStreamer\pyaxistreamer-master\outputdata_coefficients\air_rates.json
E/N: 0 ～ 2000 Td (258点)


In [4]:
OUTPUT_DIR = OUTPUT_BASE
if any(OUTPUT_DIR.glob('*.csv')):
    OUTPUT_DIR = Path(mkdtemp(prefix=f'{OUTPUT_BASE.name}_', dir=OUTPUT_BASE.parent))
print('CSV出力先:', OUTPUT_DIR, flush=True)
state = run_simulation(config, output_dir=OUTPUT_DIR)
print('maximum electron density [m^-3]:', state.ne.max())

CSV出力先: c:\Users\daiko\Documents\Python_venvSystem\3.14system\PyAxiStreamer\pyaxistreamer-master\outputdata_chemistry2_ia88nm8i
step = 104, Time = 0.100000 ns, dt = 5.000000e-13 s
step = 208, Time = 0.200000 ns, dt = 5.000000e-13 s
step = 312, Time = 0.300000 ns, dt = 5.000000e-13 s
step = 416, Time = 0.400000 ns, dt = 5.000000e-13 s
step = 520, Time = 0.500000 ns, dt = 5.000000e-13 s
step = 624, Time = 0.600000 ns, dt = 5.000000e-13 s
step = 728, Time = 0.700000 ns, dt = 5.000000e-13 s
step = 832, Time = 0.800000 ns, dt = 5.000000e-13 s
step = 937, Time = 0.900000 ns, dt = 6.319882e-14 s
step = 1074, Time = 1.000000 ns, dt = 2.604513e-13 s
step = 1258, Time = 1.100000 ns, dt = 3.942980e-13 s
step = 1481, Time = 1.200000 ns, dt = 1.199427e-13 s
step = 1737, Time = 1.300000 ns, dt = 7.374195e-14 s
step = 2026, Time = 1.400000 ns, dt = 2.672477e-13 s
step = 2415, Time = 1.500000 ns, dt = 1.146433e-13 s


TimestepError: simulation stopped at t=1.526413642e-09, step=2647: E/N 2006.44 Td exceeds table maximum 2000 Td; regenerate the table for this range

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
fields = [(state.ne, 'Electron density [m$^{-3}$]'),
          (state.phi, 'Potential [V]'), (state.abs_e, '|E|/N [Td]')]
for ax, (values, title) in zip(axes, fields):
    image = ax.pcolormesh(state.rh * 1e3, state.zh * 1e3, values.T, shading='nearest', cmap='jet')
    ax.set(xlabel='r [mm]', ylabel='z [mm]', title=title)
    fig.colorbar(image, ax=ax)
plt.show()
# 必要な場合: fig.savefig(ROOT / 'final_state.png', dpi=160)

描画は最終時刻の電子密度・電位・電場を使用します。動画を作る場合は、描画Notebookの入力先を実行セルに表示された `OUTPUT_DIR` に合わせてください。